# Track B, step 3c: fine-tuning v2, trained on the question we're asked

Step 3's text model was trained on a looser question than the competition asks, and it learned a misleading shortcut. v2 fixes both. **The two changes were decided before running v2, and every other setting is identical to step 3.**

**Fix 1: count only "exploited soon" as positive.** In step 3, a training row was positive if CISA added it to KEV *at any point* by the deadline. 54% of those 684 positives took more than a year, and 257 came from KEV's one-time launch batch on November 3, 2021. The competition asks about the next few months: 2025 validation positives took a median of 1.5 months. v2 counts a row as positive only if it reached KEV **within 12 months of publication**: 314 positives. The slower ones are **left out of training**, because calling them "not exploited" would be false.

**Fix 2: the description only, without the assigner's name.** Step 3's input began with who filed the CVE, e.g. `disclosure@vulncheck.com | ...`. In training, VulnCheck filed just 48 CVEs, 4 of them exploited, so the model learned "VulnCheck = dangerous". In 2025, VulnCheck filed 402, many of them new IDs for decade-old products, and step 3 put 95 of them in its top 243 (6 of the 402 were exploited). Track A's model already handles vendor and assigner history properly (point-in-time), so the text model should read only the description.

**How we'll judge it, with a caution.** We found fix 2 by looking at step 3's 2025 mistakes, so a 2025 improvement will look a bit better than it really is. So v2 must **also** beat step 3's recipe on a second, untouched test: train both recipes on 2018-2023 and test on 2024 (Track A's 2024 set: 28,839 rows, 33 positives). v2 counts as a win only if both tests agree.

**After this, no more model tuning:** each extra experiment uses up the 52 validation positives a little more.

## 1. Setup and settings

All settings are fixed here, before seeing any 2025 results, and follow the team plan. The 2025 validation set has only 52 positives, so we don't tune on it beyond a handful of runs.

In [1]:
import os, re, time, math, random, warnings
warnings.filterwarnings("ignore")
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import roc_auc_score
from transformers import AutoModelForSequenceClassification, AutoTokenizer, get_linear_schedule_with_warmup
from transformers.utils import logging as hf_logging
hf_logging.set_verbosity_error()

DATA = next(p for p in [Path("../data"), Path("data")] if p.exists()).resolve()
OUT = DATA / "trackb"
OUT.mkdir(exist_ok=True)
DEVICE = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
SMOKE = os.environ.get("SMOKE") == "1"

CFG = dict(
    model="ibm-granite/granite-embedding-small-english-r2",
    max_length=256,        # covers 97% of descriptions in full (step 1)
    neg_per_pos=20,        # keep all positives + 20 random negatives each
    pos_weight=4.0,        # a missed positive costs 4x a false alarm in the loss
    lr=3e-5, batch_size=32, epochs=3, warmup=0.06, weight_decay=0.01,
    holdout=0.10,          # share of training rows held back to watch the loss
    seed=0,
)
if SMOKE:
    CFG.update(epochs=1, neg_per_pos=5)

def seed_everything(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
seed_everything(CFG["seed"])
print("device:", DEVICE, "| smoke test:", SMOKE)
print(CFG)

device: mps | smoke test: False
{'model': 'ibm-granite/granite-embedding-small-english-r2', 'max_length': 256, 'neg_per_pos': 20, 'pos_weight': 4.0, 'lr': 3e-05, 'batch_size': 32, 'epochs': 3, 'warmup': 0.06, 'weight_decay': 0.01, 'holdout': 0.1, 'seed': 0}


## 2. Load the texts

Same rules as step 1: training is published 2018-2024 with labels by December 31, 2024; validation is the Phase 3 2025 set; scoring is the 2026 scoreboard. The input is `assigner | description`, with "exploited in the wild" wording removed.

The first run builds `data/trackb/texts.parquet` from the CSVs. Later runs, including on a GPU server, need only that one ~100 MB file.

In [2]:
TEXTS = OUT / "texts.parquet"
IN_WILD = re.compile(r"exploited in the wild|actively exploited|may have been exploited|aware of (?:a )?report",
                     re.IGNORECASE)

def clean(description):
    text = re.sub(r",?\s*as (?:actively )?exploited in the wild[^.]*", "", description or "", flags=re.IGNORECASE)
    sentences = re.split(r"(?<=[.!?])\s+", text)
    return " ".join(s for s in sentences if not IN_WILD.search(s)).strip()

if not TEXTS.exists():
    import duckdb
    con = duckdb.connect()
    for t in ["vulnerabilities", "known_exploited", "scoreboard_2026"]:
        con.sql(f'''CREATE VIEW "{t}" AS
                    SELECT * FROM read_csv('{DATA}/{t}.csv', quote='"', escape='"', sample_size=-1)''')
    train = con.sql('''
        WITH v AS (SELECT *, CAST(published AS DATE) AS pub FROM vulnerabilities),
             k AS (SELECT cve_id, MIN(CAST(date_added AS DATE)) AS date_added FROM known_exploited GROUP BY cve_id)
        SELECT v.cve_id, v.assigner, v.description, year(v.pub) AS year, v.pub AS published,
               k.date_added AS kev_date,
               CAST(k.date_added IS NOT NULL AND k.date_added <= DATE '2024-12-31' AS INT) AS y
        FROM v LEFT JOIN k USING (cve_id)
        WHERE v.pub BETWEEN DATE '2018-01-01' AND DATE '2024-12-31'
          AND (k.date_added IS NULL OR k.date_added > last_day(v.pub))
        ORDER BY v.cve_id''').df().assign(split="train")
    val = (pd.read_parquet(DATA / "backtest/val_2025.parquet")
             [["cve_id", "assigner", "description", "y", "epss", "any_exploit", "published", "kev_date_added"]]
             .rename(columns={"kev_date_added": "kev_date"}).assign(split="val", year=2025))
    s26 = con.sql('''SELECT s.cve_id, v.assigner, v.description FROM scoreboard_2026 s
                     LEFT JOIN vulnerabilities v USING (cve_id) ORDER BY s.cve_id''').df().assign(split="score26", year=2026)
    texts = pd.concat([train, val, s26], ignore_index=True)
    texts["published"] = pd.to_datetime(texts["published"])
    texts["kev_date"] = pd.to_datetime(texts["kev_date"])  # kept so later steps can relabel with a later deadline
    texts["text"] = texts["assigner"].fillna("unknown") + " | " + texts["description"].fillna("").map(clean)
    texts.drop(columns=["description"]).to_parquet(TEXTS, index=False)
    print("built", TEXTS.name)

texts = pd.read_parquet(TEXTS)
texts["y"] = texts["y"].fillna(0).astype(int)  # 2026 rows have no label; 0 is a placeholder, never used
train = texts[texts.split == "train"].sort_values("cve_id", ignore_index=True)
val = texts[texts.split == "val"].sort_values("cve_id", ignore_index=True)
score26 = texts[texts.split == "score26"].sort_values("cve_id", ignore_index=True)
print(f"train {len(train):,} rows / {int(train.y.sum())} positives | val {len(val):,} / {int(val.y.sum())} | "
      f"2026 {len(score26):,}")
assert not texts["text"].str.contains(IN_WILD).any()

# ---- v2 changes (decided before this run) ----
assert texts["text"].str.contains(" | ", regex=False).all()
texts["text_v1"] = texts["text"]                                      # step 3's input: assigner | description
texts["text"] = texts["text"].str.split(" | ", n=1, regex=False).str[1]   # fix 2: description only
train = texts[texts.split == "train"].sort_values("cve_id", ignore_index=True)
val = texts[texts.split == "val"].sort_values("cve_id", ignore_index=True)
score26 = texts[texts.split == "score26"].sort_values("cve_id", ignore_index=True)

def soon_labels(df, deadline, months=12):
    # fix 1: positive = on KEV within `months` of publication (and by the deadline); slower positives dropped
    delay_days = (df["kev_date"] - df["published"]).dt.days
    on_kev = df["kev_date"].notna() & (df["kev_date"] <= pd.Timestamp(deadline))
    fast = on_kev & (delay_days <= months * 30.44)
    out = df[~(on_kev & ~fast)].copy()
    out["y"] = fast[~(on_kev & ~fast)].astype(int)
    return out.reset_index(drop=True)

n_before = int(train.y.sum())
train = soon_labels(train, "2024-12-31")
print(f"v2 training set: {len(train):,} rows, {int(train.y.sum())} positives "
      f"({n_before - int(train.y.sum())} slower positives left out)")
print("example input:", train.loc[train.y == 1, "text"].iloc[0][:120], "...")

train 165,885 rows / 684 positives | val 35,281 / 52 | 2026 72,670


v2 training set: 165,515 rows, 314 positives (370 slower positives left out)
example input: The SolarWinds Orion API is vulnerable to an authentication bypass that could allow a remote attacker to execute API com ...


The Phase 3 scoring helpers: `hits_at_k`, `bootstrap` (90% range) and `compare` (paired test; a gain is real when its range excludes 0).

In [3]:
BENCH_K, BENCH_N = 500, 72_670
default_k = lambda n: int(round(n * BENCH_K / BENCH_N))

def rank_order(df, score, seed=0):
    cols = [score] if isinstance(score, str) else score
    if isinstance(cols, list) and all(isinstance(c, str) for c in cols):
        keys = [df[c].to_numpy(dtype=float) for c in cols]
    else:
        keys = [np.asarray(score, dtype=float)]
    keys = [np.where(np.isnan(k), -np.inf, k) for k in keys]
    tie = np.random.default_rng(seed).permutation(len(df))
    return np.lexsort([tie] + [-k for k in reversed(keys)])

def hits_at_k(df, score, k=None, seed=0):
    k = k or default_k(len(df))
    return int(df["y"].to_numpy()[rank_order(df, score, seed)[:k]].sum())

def _ranks(df, score, seed=0):
    r = np.empty(len(df), dtype=np.int64)
    r[rank_order(df, score, seed)] = np.arange(len(df))
    return r

def _boot_hits(ranks, y, k, idx):
    return np.array([y[i][np.argpartition(ranks[i], k)[:k]].sum() for i in idx])

def bootstrap(df, score, k=None, n_boot=1000, ci=0.90, seed=0):
    k = k or default_k(len(df))
    idx = np.random.default_rng(seed).integers(0, len(df), size=(n_boot, len(df)))
    h = _boot_hits(_ranks(df, score, seed), df["y"].to_numpy(), k, idx)
    return int(np.quantile(h, (1 - ci) / 2)), int(np.quantile(h, 1 - (1 - ci) / 2))

def compare(df, score_a, score_b, k=None, n_boot=1000, ci=0.90, seed=0):
    k = k or default_k(len(df))
    idx = np.random.default_rng(seed).integers(0, len(df), size=(n_boot, len(df)))
    y = df["y"].to_numpy()
    d = (_boot_hits(_ranks(df, score_a, seed), y, k, idx)
         - _boot_hits(_ranks(df, score_b, seed), y, k, idx))
    return {"diff": hits_at_k(df, score_a, k, seed) - hits_at_k(df, score_b, k, seed),
            "range": (int(np.quantile(d, (1 - ci) / 2)), int(np.quantile(d, 1 - (1 - ci) / 2))),
            "p_a_better": round(float((d > 0).mean()), 3)}

BASELINE = ["any_exploit", "epss"]

## 3. The fine-tuning set: downsample the negatives

Training on all 165,885 rows would take hours, and 99.6% of them are negatives, so the model would mostly learn to say "no." Instead we keep **all 684 positives plus 20 random negatives for each** (about 14,000 rows). Then 10% is held out to watch the loss.

The **positive weight of 4** in the loss means missing an exploited bug costs four times as much as a false alarm. Only the ranking order matters, so the probabilities don't need to be calibrated.

In [4]:
rng = np.random.default_rng(CFG["seed"])
pos_idx = np.flatnonzero(train.y.to_numpy() == 1)
neg_idx = np.flatnonzero(train.y.to_numpy() == 0)
if SMOKE:
    pos_idx = rng.choice(pos_idx, 64, replace=False)
neg_pick = rng.choice(neg_idx, CFG["neg_per_pos"] * len(pos_idx), replace=False)
ft = train.iloc[np.concatenate([pos_idx, neg_pick])].sample(frac=1, random_state=CFG["seed"]).reset_index(drop=True)
n_hold = int(len(ft) * CFG["holdout"])
hold, fit = ft.iloc[:n_hold].reset_index(drop=True), ft.iloc[n_hold:].reset_index(drop=True)
print(f"fine-tuning rows: {len(fit):,} ({int(fit.y.sum())} positives) | held out: {len(hold):,} ({int(hold.y.sum())} positives)")

fine-tuning rows: 5,935 (280 positives) | held out: 659 (34 positives)


## 4. The model and the training loop

`AutoModelForSequenceClassification` puts a new two-class layer on top of Granite. The loop below is plain PyTorch, so every step from the list at the top is visible:

- **AdamW** is the optimizer that applies the updates.
- The **learning rate** warms up over the first 6% of steps, then falls linearly to 0. Big early steps could wreck what Granite already knows.
- **Gradient clipping** caps any single update.
- On an NVIDIA GPU, **bfloat16** half-precision roughly doubles speed. On a Mac we stay in full precision.

In [5]:
tok = AutoTokenizer.from_pretrained(CFG["model"])

def batches(df, batch_size, shuffle=False, seed=0):
    order = np.random.default_rng(seed).permutation(len(df)) if shuffle else np.arange(len(df))
    for i in range(0, len(order), batch_size):
        rows = df.iloc[order[i:i + batch_size]]
        enc = tok(rows["text"].tolist(), truncation=True, max_length=CFG["max_length"], padding=True, return_tensors="pt")
        yield {k: v.to(DEVICE) for k, v in enc.items()}, torch.tensor(rows["y"].fillna(0).to_numpy().astype(np.int64), device=DEVICE)

use_bf16 = DEVICE == "cuda" and torch.cuda.is_bf16_supported()
autocast = (lambda: torch.autocast("cuda", dtype=torch.bfloat16)) if use_bf16 else (lambda: torch.autocast("cpu", enabled=False))

@torch.no_grad()
def predict(model, df, batch_size=128):
    # P(exploited) for every row. Sorting by length first means less padding, so it's faster.
    model.eval()
    order = df["text"].str.len().sort_values().index.to_numpy()
    out = np.empty(len(df), dtype=np.float32)
    sorted_df = df.loc[order]
    i = 0
    for enc, _ in batches(sorted_df.assign(y=0), batch_size):
        with autocast():
            logits = model(**enc).logits.float()
        p = torch.softmax(logits, dim=-1)[:, 1].cpu().numpy()
        out[i:i + len(p)] = p
        i += len(p)
    result = np.empty(len(df), dtype=np.float32)
    result[np.searchsorted(df.index.to_numpy(), order)] = out
    return result

def train_model(fit, hold, label="model"):
    seed_everything(CFG["seed"])
    model = AutoModelForSequenceClassification.from_pretrained(CFG["model"], num_labels=2).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=CFG["lr"], weight_decay=CFG["weight_decay"])
    steps = CFG["epochs"] * math.ceil(len(fit) / CFG["batch_size"])
    sched = get_linear_schedule_with_warmup(opt, int(CFG["warmup"] * steps), steps)
    loss_fn = torch.nn.CrossEntropyLoss(weight=torch.tensor([1.0, CFG["pos_weight"]], device=DEVICE))
    log = []
    for epoch in range(CFG["epochs"]):
        model.train()
        t0, total, n = time.time(), 0.0, 0
        for enc, y in batches(fit, CFG["batch_size"], shuffle=True, seed=CFG["seed"] + epoch):
            with autocast():
                loss = loss_fn(model(**enc).logits.float(), y)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step(); sched.step(); opt.zero_grad()
            total += loss.item() * len(y); n += len(y)
        p_hold = predict(model, hold)
        p = np.clip(p_hold.astype(np.float64), 1e-7, 1 - 1e-7)  # weighted log-loss on the held-out rows, on the CPU
        w = np.where(hold.y.to_numpy() == 1, CFG["pos_weight"], 1.0)
        hold_loss = float(-(w * np.where(hold.y.to_numpy() == 1, np.log(p), np.log(1 - p))).sum() / w.sum())
        log.append({"epoch": epoch + 1, "train loss": round(total / n, 4), "held-out loss": round(hold_loss, 4),
                    "held-out AUC": round(float(roc_auc_score(hold.y, p_hold)), 3), "minutes": round((time.time() - t0) / 60, 1)})
        print(f"[{label}]", log[-1])
    return model, pd.DataFrame(log)

## 5. Train

In [6]:
t_start = time.time()
model, train_log = train_model(fit, hold, "v2")
print(f"total training time: {(time.time() - t_start) / 60:.1f} min")
train_log

[v2] {'epoch': 1, 'train loss': 0.3959, 'held-out loss': 0.477, 'held-out AUC': 0.86, 'minutes': 5.3}


[v2] {'epoch': 2, 'train loss': 0.204, 'held-out loss': 0.3885, 'held-out AUC': 0.869, 'minutes': 5.4}


[v2] {'epoch': 3, 'train loss': 0.111, 'held-out loss': 0.5455, 'held-out AUC': 0.866, 'minutes': 5.6}
total training time: 16.3 min


,epoch,train loss,held-out loss,held-out AUC,minutes
0,1,0.3959,0.4770,0.860,5.3
1,2,0.2040,0.3885,0.869,5.4
2,3,0.1110,0.5455,0.866,5.6


**How to read the training log:** the *training* loss should fall every epoch. The *held-out* loss and AUC are for rows the model never trained on, so they show whether it's learning patterns or memorizing. If held-out loss rises while training loss falls, the model is overfitting.

## 6. Score the 2025 validation set

This is the real test: 35,281 vulnerabilities the model has never seen, published after everything it trained on.

In [7]:
eval_val = val.sample(3000, random_state=0).sort_values("cve_id", ignore_index=True) if SMOKE else val
t0 = time.time()
s_ft = predict(model, eval_val)
print(f"scored {len(eval_val):,} rows in {(time.time() - t0) / 60:.1f} min")

v1 = pd.read_csv(OUT / "text_val_2025_finetuned.csv").set_index("cve_id")["text_score"].reindex(eval_val.cve_id).to_numpy()
frozen = pd.read_csv(OUT / "text_val_2025_frozen.csv").set_index("cve_id")["text_score"].reindex(eval_val.cve_id).to_numpy()
rows = []
for name, s in [("Baseline: exploit code then EPSS", BASELINE), ("Frozen Granite (step 2)", frozen),
                ("Fine-tuned v1 (step 3)", v1), ("Fine-tuned v2 (this run)", s_ft)]:
    r = {"ranking": name, "hits@k": hits_at_k(eval_val, s), "90% range": "%d-%d" % bootstrap(eval_val, s)}
    if not isinstance(s, list):
        r["AUC"] = round(roc_auc_score(eval_val.y, s), 3)
    rows.append(r)
print("k =", default_k(len(eval_val)))
pd.DataFrame(rows).fillna("")

scored 35,281 rows in 6.1 min


k = 243


,ranking,hits@k,90% range,AUC
0,Baseline: exploit code then EPSS,14,8-21,
1,Frozen Granite (step 2),4,1-8,0.8
2,Fine-tuned v1 (step 3),5,2-9,0.854
3,Fine-tuned v2 (this run),2,0-5,0.847


### Paired comparisons and the wider view

- **Fine-tuned vs. frozen:** did fine-tuning sharpen the text score?
- **Fine-tuned vs. baseline:** how does text alone compare with the best signals ranking?
- **Hits at larger cut-offs:** where in the ranking the gains come from.

In [8]:
print("v2 vs. v1:       ", compare(eval_val, s_ft, v1))
print("v2 vs. baseline: ", compare(eval_val, s_ft, BASELINE))
top = rank_order(eval_val, s_ft)[:default_k(len(eval_val))]
old_id = (pd.to_datetime(eval_val["published"]).dt.year - eval_val["cve_id"].str.slice(4, 8).astype(int)) >= 2
print(f"v2 top {len(top)}: {int(old_id.iloc[top].sum())} old-ID rows, {int((eval_val['assigner'].iloc[top] == 'disclosure@vulncheck.com').sum())} VulnCheck rows")
pd.DataFrame([{"top k": k, "v2": hits_at_k(eval_val, s_ft, k), "v1": hits_at_k(eval_val, v1, k), "frozen": hits_at_k(eval_val, frozen, k),
               "EPSS first": hits_at_k(eval_val, "epss", k), "exploit code then EPSS": hits_at_k(eval_val, BASELINE, k)}
              for k in [243, 500, 1000, 2000, 5000] if k < len(eval_val)])

v2 vs. v1:        {'diff': -3, 'range': (-6, -1), 'p_a_better': 0.0}


v2 vs. baseline:  {'diff': -12, 'range': (-19, -7), 'p_a_better': 0.0}
v2 top 243: 14 old-ID rows, 42 VulnCheck rows


,top k,v2,v1,frozen,EPSS first,exploit code then EPSS
0,243,2,5,4,8,14
1,500,3,8,6,8,18
2,1000,9,11,9,13,22
3,2000,16,23,19,18,24
4,5000,31,35,26,28,33


**Reading this: on 2025, v2 is worse than v1.** It finds 2 positives in the top 243, against v1's 5: a paired difference of −3, range −6 to −1, so the loss is real. AUC also slips, from 0.854 to 0.847. The fixes did what they were meant to (old-ID rows in the top 243 fell from 45 to 14, and VulnCheck rows from 95 to 42), but the hits fell too. v2 also trails v1 at every deeper cut (16 vs. 23 in the top 2,000, 31 vs. 35 in the top 5,000).

### Blend with the signals baseline

The same rank average as step 2, with weights fixed in advance: 50/50 and 70/30 (baseline/text).

In [9]:
rank_pct = lambda df, s: _ranks(df, s) / (len(df) - 1)  # 0 = best
rows = []
for w in [0.5, 0.7]:
    for name, s in [("frozen", frozen), ("fine-tuned", s_ft)]:
        b = -(w * rank_pct(eval_val, BASELINE) + (1 - w) * rank_pct(eval_val, s))
        c = compare(eval_val, b, BASELINE)
        rows.append({"blend (baseline/text)": f"{w:.0%} / {1 - w:.0%}", "text": name, "hits@k": hits_at_k(eval_val, b),
                     "vs. baseline": c["diff"], "diff range": "%+d to %+d" % c["range"], "P(better)": c["p_a_better"]})
pd.DataFrame(rows)

,blend (baseline/text),text,hits@k,vs. baseline,diff range,P(better)
0,50% / 50%,frozen,11,-3,-9 to +2,0.124
1,50% / 50%,fine-tuned,9,-5,-10 to -1,0.005
2,70% / 30%,frozen,14,0,-7 to +4,0.305
3,70% / 30%,fine-tuned,11,-3,-8 to +1,0.060


### What the description knew before EPSS did

These are exploited vulnerabilities that the fine-tuned model ranks highly while EPSS ranks them 5,000th or lower.

In [10]:
ranked = eval_val.assign(text_rank=_ranks(eval_val, s_ft), epss_rank=_ranks(eval_val, "epss"))
pos = ranked[ranked.y == 1]
for top in [243, 2000]:
    print(f"positives in fine-tuned top {top:,} that EPSS ranks 5,000th or lower: "
          f"{((pos.text_rank < top) & (pos.epss_rank >= 5000)).sum()}")
pos.sort_values("text_rank")[["cve_id", "text_rank", "epss_rank", "text"]].head(15)

positives in fine-tuned top 243 that EPSS ranks 5,000th or lower: 0
positives in fine-tuned top 2,000 that EPSS ranks 5,000th or lower: 5


,cve_id,text_rank,epss_rank,text
20621,CVE-2025-34028,16,145,The Commvault Command Center Innovation Releas...
19928,CVE-2025-32433,39,165,Erlang/OTP is a set of libraries for the Erlan...
27915,CVE-2025-5086,457,29,A deserialization of untrusted data vulnerabil...
3754,CVE-2024-13161,706,11249,Absolute path traversal in Ivanti EPM before t...
3752,CVE-2024-13159,707,12913,Absolute path traversal in Ivanti EPM before t...
3753,CVE-2024-13160,708,13302,Absolute path traversal in Ivanti EPM before t...
22509,CVE-2025-3935,740,1722,ScreenConnect versions 25.2.3 and earlier vers...
26674,CVE-2025-48703,888,2030,CWP (aka Control Web Panel or CentOS Web Panel...
19966,CVE-2025-3248,988,3,Langflow versions prior to 1.3.0 are susceptib...
32530,CVE-2025-6554,1069,33253,Type confusion in V8 in Google Chrome prior to...


## 7. Save

- `text_val_2025_v2.csv`: the team handoff format (`cve_id, text_score`) for the Compare and combine table
- the fine-tuned model in `data/trackb/finetuned_v2/`, for scoring 2026 and the 5-fold step (B4)

A smoke test saves nothing, so it never overwrites a real run.

In [11]:
if SMOKE:
    print("Smoke test: nothing saved.")
else:
    pd.DataFrame({"cve_id": eval_val["cve_id"], "text_score": s_ft}).to_csv(OUT / "text_val_2025_v2.csv", index=False)
    model.save_pretrained(OUT / "finetuned_v2"); tok.save_pretrained(OUT / "finetuned_v2")
    train_log.to_csv(OUT / "finetune_log_v2.csv", index=False)
    print("Saved:", *sorted(p.name for p in OUT.iterdir()))

Saved: emb_score26.npy emb_score26_ids.parquet emb_train.npy emb_train_ids.parquet emb_val.npy emb_val_ids.parquet finetune_log.csv finetune_log_v2.csv finetuned finetuned_v2 handoff handoff_frozen kev_similarity_2026.csv kev_similarity_val_2025.csv text_val_2025_finetuned.csv text_val_2025_frozen.csv text_val_2025_v2.csv texts.parquet


## 8. The second, untouched test: 2018-2023 → 2024

Both recipes are trained on vulnerabilities published 2018-2023, with labels by December 31, **2023**, so nothing from 2024 leaks in. Then both are tested on Track A's 2024 set: published January-September 2024, positive if added to KEV by December 31, 2024, 33 positives, scored in the top 198 (the same 0.69% cut).

- **v1 recipe (step 3):** `assigner | description`, positive if on KEV at any point by the deadline
- **v2 recipe:** description only, positive only if on KEV within 12 months, slower positives left out

Same model, same settings, same seed. Only the two fixes differ.

In [12]:
test24 = (pd.read_parquet(DATA / "backtest/train_2024.parquet")[["cve_id", "y", "epss", "any_exploit"]]
          .merge(texts[["cve_id", "text", "text_v1"]], on="cve_id", how="left")
          .sort_values("cve_id", ignore_index=True))
assert test24["text"].notna().all()
if SMOKE:
    test24 = test24.sample(800, random_state=0).sort_values("cve_id", ignore_index=True)
pre = texts[(texts.split == "train") & (texts.year <= 2023)].sort_values("cve_id", ignore_index=True).copy()
pre["y"] = (pre["kev_date"].notna() & (pre["kev_date"] <= pd.Timestamp("2023-12-31"))).astype(int)

def fit_predict(rows, test_text, label):
    rng = np.random.default_rng(CFG["seed"])
    pos, neg = np.flatnonzero(rows.y.to_numpy() == 1), np.flatnonzero(rows.y.to_numpy() == 0)
    if SMOKE:
        pos = rng.choice(pos, min(64, len(pos)), replace=False)
    ft = rows.iloc[np.concatenate([pos, rng.choice(neg, CFG["neg_per_pos"] * len(pos), replace=False)])]
    ft = ft.sample(frac=1, random_state=CFG["seed"]).reset_index(drop=True)
    n_hold = int(len(ft) * CFG["holdout"])
    m, _ = train_model(ft.iloc[n_hold:].reset_index(drop=True), ft.iloc[:n_hold].reset_index(drop=True), label)
    s = predict(m, test_text)
    del m
    return s

t0 = time.time()
v1_rows = pre.assign(text=pre["text_v1"])
s24_v1 = fit_predict(v1_rows, test24.assign(text=test24["text_v1"]), "2024 check, v1 recipe")
v2_rows = soon_labels(pre, "2023-12-31")
s24_v2 = fit_predict(v2_rows, test24, "2024 check, v2 recipe")
print(f"2024 check took {(time.time() - t0) / 60:.1f} min | training positives: v1 {int(v1_rows.y.sum())}, v2 {int(v2_rows.y.sum())}")

k24 = default_k(len(test24))
res = pd.DataFrame([
    {"ranking": "Baseline: exploit code then EPSS", f"hits@{k24}": hits_at_k(test24, BASELINE)},
    {"ranking": "v1 recipe (step 3)", f"hits@{k24}": hits_at_k(test24, s24_v1), "AUC": round(float(roc_auc_score(test24.y, s24_v1)), 3)},
    {"ranking": "v2 recipe", f"hits@{k24}": hits_at_k(test24, s24_v2), "AUC": round(float(roc_auc_score(test24.y, s24_v2)), 3)},
])
print("v2 vs. v1 on 2024:", compare(test24, s24_v2, s24_v1))
for k in [500, 1000, 2000]:
    print(f"  top {k}: v2 {hits_at_k(test24, s24_v2, k)} | v1 {hits_at_k(test24, s24_v1, k)} | baseline {hits_at_k(test24, BASELINE, k)}")
res.fillna("")

[2024 check, v1 recipe] {'epoch': 1, 'train loss': 0.3916, 'held-out loss': 0.3038, 'held-out AUC': 0.89, 'minutes': 11.0}


[2024 check, v1 recipe] {'epoch': 2, 'train loss': 0.227, 'held-out loss': 0.2768, 'held-out AUC': 0.918, 'minutes': 81.0}


[2024 check, v1 recipe] {'epoch': 3, 'train loss': 0.1358, 'held-out loss': 0.3494, 'held-out AUC': 0.925, 'minutes': 18.9}


[2024 check, v2 recipe] {'epoch': 1, 'train loss': 0.4148, 'held-out loss': 0.3944, 'held-out AUC': 0.806, 'minutes': 3.6}


[2024 check, v2 recipe] {'epoch': 2, 'train loss': 0.2218, 'held-out loss': 0.3825, 'held-out AUC': 0.867, 'minutes': 3.7}


[2024 check, v2 recipe] {'epoch': 3, 'train loss': 0.1084, 'held-out loss': 0.5276, 'held-out AUC': 0.867, 'minutes': 3.9}


2024 check took 129.9 min | training positives: v1 593, v2 258


v2 vs. v1 on 2024: {'diff': -2, 'range': (-4, 1), 'p_a_better': 0.052}
  top 500: v2 9 | v1 8 | baseline 10
  top 1000: v2 13 | v1 13 | baseline 12
  top 2000: v2 18 | v1 19 | baseline 13


,ranking,hits@198,AUC
0,Baseline: exploit code then EPSS,7,
1,v1 recipe (step 3),6,0.896
2,v2 recipe,4,0.85


**Reading this: the untouched 2024 test agrees.** v2 finds 4 of 33 in the top 198, against v1's 6 (paired −2, range −4 to +1), and its AUC is lower (0.850 vs. 0.896). Both recipes trail the baseline (7). The two tests point the same way.

## Verdict: v2 is not adopted

v2 had to beat v1 on **both** tests, and it lost both: 2025 (2 vs. 5 hits; paired range −6 to −1) and the untouched 2024 test (4 vs. 6; −4 to +1). **v1 (step 3) remains the text model.**

**Why the fixes backfired** (the likely reasons):
1. **Fewer examples:** counting only "exploited within 12 months" cut the training positives from 684 to 314. With so few positives, the extra data mattered more than its label being perfectly matched.
2. **The assigner carried real signal as well as the shortcut:** VulnCheck CVEs in 2025 were still exploited at 10x the base rate (6 of 402). Removing the name removed that signal too.

**As agreed, model tuning stops here.** For the write-up: *"We tested a recipe that trained on the competition's exact question and removed a misleading shortcut. It fixed the symptom (far fewer old-ID and VulnCheck rows at the top) but lost hits on two independent tests, so we kept the original."*